###**Parameters**

###**Fetching parameters & creating variables**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# #catalog
# catalog="workspace"

# #Key columns list
# key_cols="['flight_id']"
# key_cols_list=eval(key_cols)

# #CDC column
# cdc_col="modifiedDate"

# #Backdated referesh
# backdated_refresh=""

# #Source object
# source_object="silver_flights"
# #source schema
# source_schema="silver"
# #target object
# target_object="dimflights"
# #target schema
# target_schema="gold"
# #surrogate key
# surrogate_key="DimFlightsKey"

In [0]:
# #catalog
# catalog="workspace"

# #Key columns list
# key_cols="['airport_id']"
# key_cols_list=eval(key_cols)

# #CDC column
# cdc_col="modifiedDate"

# #Backdated referesh
# backdated_refresh=""

# #Source object
# source_object="silver_airports"
# #source schema
# source_schema="silver"
# #target object
# target_object="dimAirports"
# #target schema
# target_schema="gold"
# #surrogate key
# surrogate_key="DimAirportsKey"

In [0]:
#catalog
catalog="workspace"

#Key columns list
key_cols="['passenger_id']"
key_cols_list=eval(key_cols)

#CDC column
cdc_col="modifiedDate"

#Backdated referesh
backdated_refresh=""

#Source object
source_object="silver_passengers"
#source schema
source_schema="silver"
#target object
target_object="dimPassengers"
#target schema
target_schema="gold"
#surrogate key
surrogate_key="DimPassengersKey"

In [0]:
key_cols_list

###**Incremental Data Ingestion**

####**Last Load**

In [0]:
#NO Backdated Refresh
if len(backdated_refresh) == 0:
  #If table existin the destination
  if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    last_load= spark.sql(f"SELECT max({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
  else:
    last_load = "1990-01-01 00:00:00"
#Yes backdated refresh    
else:
  last_load=backdated_refresh


In [0]:
last_load

In [0]:
df_src = spark.sql(
    f"SELECT * FROM {source_schema}.{source_object} WHERE {cdc_col} > '{last_load}'")


In [0]:
df_src.display()

#### **OLd vs New Records**

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    # Key columns string for incremental
    key_cols_string_incremental = ",".join(key_cols_list)

    df_trg = spark.sql(f"""
        SELECT 
            {key_cols_string_incremental}, 
            {surrogate_key},
            create_date, 
            update_date 
        FROM {catalog}.{target_schema}.{target_object}
    """)

else:
    # Key columns string for initial
    key_cols_string_init = ", ".join(f"'' AS {i}" for i in key_cols_list)

    df_trg = spark.sql(f"""
        SELECT 
            {key_cols_string_init}, 
            CAST('0' AS INT) AS {surrogate_key},
            CAST('1990-01-01 00:00:00' AS timestamp ) AS create_date, 
            CAST('1990-01-01 00:00:00' AS timestamp ) AS update_date
        WHERE 1=0
    """)

In [0]:
df_trg.display()

In [0]:
spark.sql("""
SELECT 
    '' AS flight_id, 
    ''  AS DimFlightsKey,
    '1990-01-01 00:00:00' AS create_date,
    '1990-01-01 00:00:00' AS update_date
FROM workspace.silver.silver_flights WHERE 1=0
""").display()

In [0]:
#join condition
join_condition=' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])
join_condition

In [0]:
df_src.createOrReplaceTempView('src')
df_trg.createOrReplaceTempView('trg')
df_join=spark.sql(f"""
           SELECT src.*,
                 trg.{surrogate_key},
                 trg.create_date,
                 trg.update_date
            FROM src
            LEFT JOIN trg
            ON {join_condition}

""")

In [0]:
df_join.display()

In [0]:
df_old=df_join.filter(col(f'{surrogate_key}').isNotNull())
df_old.display()
df_new=df_join.filter(col(f'{surrogate_key}').isNull()   )
df_new.display()


####**Preparing DF_OLD**

In [0]:
#enriched version of df_old
df_old_enr=df_old.withColumn('update_date',current_timestamp())


####**Preparing DF_NEW**

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
  max_surrogate_key=spark.sql(f"""
          SELECT MAX({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}
""").collect()[0][0]
  df_new_enr=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
  .withColumn('create_date',current_timestamp())\
  .withColumn('update_date',current_timestamp())
  
else:
  max_surrogate_key =0
  df_new_enr=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
  .withColumn('create_date',current_timestamp())\
  .withColumn('update_date',current_timestamp())


  


In [0]:
df_new_enr.display()

In [0]:
df_old_enr.display()

#### **Union OLD and NEW Records**

In [0]:
df_union=df_old_enr.unionByName(df_new_enr)
df_union.display()
 

####**UPSERT**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    
    dlt_obj.alias('trg').merge(
        df_union.alias('src'),
        f"trg.{surrogate_key} = src.{surrogate_key}"
    ) \
    .whenMatchedUpdateAll(
        condition=f"src.{cdc_col} >= trg.{cdc_col}"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

else:
    df_union.write.format('delta') \
        .mode('append') \
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")
                           

In [0]:
%sql
SELECT* FROM gold.dimPassengers where passenger_id='P0049'